# BioViL-T Backbone Verification

**Model**: `microsoft/BiomedVLP-BioViL-T`  
**Arch**: Hybrid ViT + ResNet-50 (via HI-ML-Multimodal)  
**Embed dim**: 128 (projected joint image-text space)  
**Input**: 480×480 px, 3-channel (grayscale → repeated)  
**Access**: HuggingFace, no auth required  
**Zero-shot**: Yes — paired text encoder via `AutoTokenizer`

Paper: [Learning to Exploit Temporal Structure for Biomedical Vision-Language Processing (CVPR 2023)](https://arxiv.org/abs/2301.04558)

In [ ]:
%pip install -q transformers torch torchvision Pillow

In [ ]:
import os

# Control which GPU(s) are visible to this notebook.
# Set CUDA_VISIBLE_DEVICES in your shell before launching Jupyter to override,
# e.g.:  CUDA_VISIBLE_DEVICES=2 jupyter lab
# Or change the default value below.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
print(f"CUDA_VISIBLE_DEVICES = {os.environ['CUDA_VISIBLE_DEVICES']}")

## 1. Load model

In [ ]:
import torch
from transformers import AutoModel, AutoTokenizer

HUB = "microsoft/BiomedVLP-BioViL-T"

model = AutoModel.from_pretrained(HUB, trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained(HUB, trust_remote_code=True)
model.eval()

print(type(model))                 # BioViLTModel
print(type(model.image_model))     # MultiImageModel (ResNet50 + BioViL projection)
print(type(model.text_model))      # BiomedVLPTextModel

## 2. Image transform

From the HI-ML-Multimodal package (`create_chest_xray_transform_for_inference`):
- Resize shorter side → 512 (BICUBIC)
- CenterCrop → 480
- ToTensor → [0, 1] single channel
- Repeat grayscale to 3 channels (ExpandChannels)
- **No** `Normalize` — model was trained without explicit mean/std normalization

In [ ]:
from torchvision import transforms
from torchvision.transforms.functional import InterpolationMode
from PIL import Image

RESIZE   = 512
CROP     = 480

def make_transform():
    return transforms.Compose([
        transforms.Resize(RESIZE, interpolation=InterpolationMode.BICUBIC),
        transforms.CenterCrop(CROP),
        transforms.Grayscale(num_output_channels=1),
        transforms.ToTensor(),   # → [1, H, W] in [0, 1]
    ])

transform = make_transform()

def preprocess(path: str) -> torch.Tensor:
    """Load CXR, return [1, 3, 480, 480] batch tensor."""
    img = Image.open(path).convert("RGB")
    x = transform(img)           # [1, 480, 480]
    x = x.repeat(3, 1, 1)        # [3, 480, 480] – ExpandChannels
    return x.unsqueeze(0)        # [1, 3, 480, 480]

# Smoke-test with random tensor
dummy = torch.rand(1, 3, CROP, CROP)
print("dummy input:", dummy.shape)

## 3. Extract image embeddings

In [ ]:
with torch.no_grad():
    output = model.image_model(pixel_values=dummy)

print("projected_global_embedding:", output.projected_global_embedding.shape)  # [1, 128]
print("patch_embeddings:           ", output.patch_embeddings.shape)            # [1, 900, D]
# patch_embeddings = projected_patch_embeddings (per-patch vectors)

emb = output.projected_global_embedding   # use this for linear probe / kNN

## Segmentation mode — spatial feature map

For dense prediction (segmentation, detection), reshape the flat patch sequence to a
2-D spatial grid. BioViL-T's `patch_embeddings` from section 3 are already [B, N, D].
The 480×480 input with 16-px patch stride gives a **30×30** patch grid.

In [ ]:
import math

# patch_embeddings: [B, N, D] — 480px / 16px stride → 30×30 = 900 patches
patches = output.patch_embeddings   # [B, 900, D]

B, N, D = patches.shape
H = W = int(math.sqrt(N))  # 30 for 480px / 16px stride
assert H * W == N

spatial = patches.permute(0, 2, 1).reshape(B, D, H, W)
print(f"patch tokens:  {patches.shape}")   # [1, 900, D]
print(f"spatial map:   {spatial.shape}")   # [1, D, 30, 30]

## 4. Zero-shot: text embeddings & similarity

In [ ]:
prompts = [
    "No pleural effusion.",
    "Large pleural effusion present.",
    "Cardiomegaly is noted.",
]

tok = tokenizer.batch_encode_plus(
    prompts, add_special_tokens=True, padding="longest", return_tensors="pt"
)

with torch.no_grad():
    text_emb = model.get_projected_text_embeddings(
        input_ids=tok.input_ids,
        attention_mask=tok.attention_mask,
    )  # [3, 128]

# L2-normalize and compute similarity
img_norm  = torch.nn.functional.normalize(emb, dim=-1)       # [1, 128]
text_norm = torch.nn.functional.normalize(text_emb, dim=-1)  # [3, 128]
sim = (img_norm @ text_norm.T).squeeze(0)                    # [3]

for p, s in zip(prompts, sim.tolist()):
    print(f"{s:.4f}  {p}")

## Notes for RadHarmony integration

```
model_call : lambda m, x: m.image_model(pixel_values=x).projected_global_embedding
pool       : None  (model_call already returns [B, 128])
embed_dim  : 128
transform  : Resize(512) → CenterCrop(480) → ToTensor → repeat(3)
```

For zero-shot, also pass `text_encoder = model.get_projected_text_embeddings` + the tokenizer.

**Segmentation mode** (`output_keys={"img", "mask"}`):
- Patch tokens: `model.image_model(pixel_values=x).patch_embeddings` → `[B, 900, D]`
- Spatial map: reshape to `[B, D, 30, 30]` (480px / 16px stride)
- Use `patches.permute(0,2,1).reshape(B, D, 30, 30)` for the segmentation head

In [ ]:
# Replace with a real CXR path
CXR_PATH = "/path/to/cxr.jpg"

x = preprocess(CXR_PATH)
with torch.no_grad():
    out = model.image_model(pixel_values=x)
    img_emb = out.projected_global_embedding  # [1, 128]

img_norm = torch.nn.functional.normalize(img_emb, dim=-1)
sim = (img_norm @ text_norm.T).squeeze(0)
for p, s in zip(prompts, sim.tolist()):
    print(f"{s:.4f}  {p}")

## Notes for RadHarmony integration

```
model_call : lambda m, x: m.image_model(pixel_values=x).projected_global_embedding
pool       : None  (model_call already returns [B, 128])
embed_dim  : 128
transform  : Resize(512) → CenterCrop(480) → ToTensor → repeat(3)
```

For zero-shot, also pass `text_encoder = model.get_projected_text_embeddings` + the tokenizer.